# Step 4: Add Metadata and Analyze Spatial Patterns


This notebook turns metric results into spatial summaries. You will use PGA and FAS as two separate examples, so you can see how the same spatial tests can highlight different model-performance patterns for different metrics.


## Imports

These functions prepare metric fields and calculate spatial summaries.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from spatial_vtk.config import (
    notebook_timer,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment(loky_max_cpu_count=1)

with notebook_timer():

    from IPython.display import Markdown, display

    from spatial_vtk.io import output_group
    from spatial_vtk.spatial import (
        run_spatial_statistics_workflow_from_config,
        spatial_workflow_failure_frame,
        summarize_standard_spatial_products,
    )
    from spatial_vtk.spatial.plot import (
        write_standard_spatial_diagnostic_figures,
        write_standard_spatial_map_figures,
    )
    register_svtk_cell_timer()


## Configuration

Load the config and read the spatial-statistics settings from the tutorial run scenario.


In [ ]:
from spatial_vtk.config import notebook_figure_settings, notebook_run_context

config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
context = notebook_run_context(config_path, run_scenario="tutorial")
cfg = context.cfg
step_outputs = output_group("step_04_spatial", cfg=cfg)

# Step 4 uses the configured larger QC-passed metric snapshot so per-metric spatial tests have enough stations.
spatial_figure_settings = notebook_figure_settings("spatial")
spatial_metrics = ("PGA", "FAS")


## Prepare Metric-Specific Spatial Fields

Spatial statistics use one value per event-station observation, plus station and event coordinates. This notebook delegates the table-building work to the package workflow, then uses the returned tables for compact displays and figures.

In [ ]:
# Run the config-backed spatial workflow and then load the standard tables it writes.
# The metrics and station metadata arguments are config keys, so the notebook does not resolve paths itself.
spatial_result = run_spatial_statistics_workflow_from_config(
    config_path=str(config_path),
    run_scenario=context.run_scenario,
    metrics="paths.metric_figure_snapshot",
    metric=spatial_metrics,
    station_metadata="paths.site_metadata",
    verbose=True,
)

failure_table = spatial_workflow_failure_frame(spatial_result)
if not failure_table.empty:
    display(Markdown("### Non-fatal workflow diagnostics"))
    display(failure_table)

# Keep small per-metric table handles for the plotting cells below.
spatial_tables = step_outputs.load_tables(
    {
        "metric_field": "metric_field_path",
        "event_centered_residuals": "event_centered_path",
        "station_bias": "station_bias_path",
        "morans_i": "morans_i_path",
        "distance_bins": "distance_corr_path",
        "clusters": "clusters_path",
        "cluster_scores": "cluster_scores_path",
        "cluster_summary": "cluster_summary_path",
        "pca_station_scores": "pca_scores_path",
        "pca_feature_loadings": "pca_loadings_path",
        "pca_explained_variance": "pca_explained_path",
        "geology_contrasts": "geology_path",
    },
    cfg=cfg,
)
metric_field = spatial_tables["metric_field"]
event_centered_residuals = spatial_tables["event_centered_residuals"]
station_bias = spatial_tables["station_bias"]
spatial_product_summary = summarize_standard_spatial_products(
    spatial_result,
    metric_field=metric_field,
    event_centered=event_centered_residuals,
    station_bias=station_bias,
)
spatial_metrics_run = spatial_product_summary.metrics
spatial_products = spatial_product_summary.spatial_products

display(spatial_product_summary.summary_frame())
display(spatial_product_summary.station_bias_preview_frame())


## Station Bias Maps

These maps show the mean event-centered residual at each station. Positive values mean the observed amplitudes are larger than the synthetic amplitudes on average for that metric.


In [ ]:
spatial_map_result = write_standard_spatial_map_figures(
    spatial_products,
    step_outputs,
    spatial_figure_settings,
)
display(spatial_map_result.status_frame())

## Residual Grid Maps

A residual grid gives you a quick spatial overview of where residuals are broadly positive or negative. These examples use the same event-centered residual field as the station-bias maps.


## Spatial Diagnostic Figures

Render the spatial-correlation, PCA-summary, and geology-contrast figures from the Step 4 output tables. The package helper owns the metric filters and figure paths, and returns the compact diagnostic rows used to annotate the figures.


In [ ]:
# Render correlation, PCA, and geology diagnostic figures from the package-owned Step 4 suite.
# The helper selects the metric-specific tables, writes configured figure paths, handles sidecars,
# and returns the compact diagnostic rows that were used to annotate the figures.
spatial_diagnostic_result = write_standard_spatial_diagnostic_figures(
    spatial_products,
    spatial_tables,
    step_outputs,
    spatial_figure_settings,
    cfg=cfg,
    metrics=spatial_metrics_run,
)
display(spatial_diagnostic_result.preview_frame())
display(spatial_diagnostic_result.status_frame())


## Spatial Figure Provenance

Review the figure sidecar metadata written for this step without loading the full row CSV sidecars.

In [ ]:
display(spatial_figure_settings.sidecars.readiness_frame())
display(spatial_figure_settings.status_frame())
